In [31]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("/home/redstevo/Documents/dataset/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df['MultipleLines'].unique()

array(['No phone service', 'No', 'Yes'], dtype=object)

In [5]:
df['InternetService'].unique()

array(['DSL', 'Fiber optic', 'No'], dtype=object)

In [6]:
df['Contract'].unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

In [7]:
df['PaymentMethod'].unique()

array(['Electronic check', 'Mailed check', 'Bank transfer (automatic)',
       'Credit card (automatic)'], dtype=object)

In [8]:
df['SeniorCitizen'].unique()

array([0, 1])

In [9]:
df['TotalCharges'] = df['TotalCharges'].convert_dtypes(convert_floating=True)

In [10]:
df.iloc[488:489]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No


In [11]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [12]:
df["SeniorCitizen"] = df["SeniorCitizen"].astype(float)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   float64
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [14]:
df.fillna( {'TotalCharges': df['TotalCharges'].mean()}, inplace=True)

In [15]:
df.iloc[488:489]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0.0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,2283.300441,No


In [16]:
transformer = ColumnTransformer(
    transformers=[
        ('one hot encoding', OneHotEncoder(sparse_output=False), ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']),
        ('standard scaling', StandardScaler(), ['tenure', 'MonthlyCharges', 'TotalCharges'])
    ]
    
)

In [17]:
transformer.set_output(transform="pandas")

,transformers,"[('one hot encoding', ...), ('standard scaling', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [18]:
x = transformer.fit_transform(df.drop(columns=["Churn"], axis=1))
x["SeniorCitizen"] = df["SeniorCitizen"]

In [19]:
x.head()

,one hot encoding__gender_Female,one hot encoding__gender_Male,one hot encoding__Partner_No,one hot encoding__Partner_Yes,one hot encoding__Dependents_No,one hot encoding__Dependents_Yes,one hot encoding__PhoneService_No,one hot encoding__PhoneService_Yes,one hot encoding__MultipleLines_No,one hot encoding__MultipleLines_No phone service,...,one hot encoding__PaperlessBilling_No,one hot encoding__PaperlessBilling_Yes,one hot encoding__PaymentMethod_Bank transfer (automatic),one hot encoding__PaymentMethod_Credit card (automatic),one hot encoding__PaymentMethod_Electronic check,one hot encoding__PaymentMethod_Mailed check,standard scaling__tenure,standard scaling__MonthlyCharges,standard scaling__TotalCharges,SeniorCitizen
0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-1.277445,-1.160323,-0.994971,0.0
1,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.066327,-0.259629,-0.173876,0.0
2,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,-1.236724,-0.362660,-0.960399,0.0
3,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.514251,-0.746535,-0.195400,0.0
4,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-1.236724,0.197365,-0.941193,0.0


In [20]:
df['Churn'] = df['Churn'].apply(lambda x : 1 if x.lower() == "yes" else 0)

y = df['Churn']

In [21]:
forest_model = RandomForestClassifier(n_estimators=300, criterion='entropy', n_jobs=40)

In [22]:
forest_model.fit(x, y)

,n_estimators,300
,criterion,'entropy'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
importance = forest_model.feature_importances_
importance

array([0.017607  , 0.01797202, 0.01471116, 0.01460812, 0.01251028,
       0.01270456, 0.00302025, 0.00297274, 0.01316745, 0.00299744,
       0.01310481, 0.01003735, 0.02064307, 0.00537329, 0.02830053,
       0.00487574, 0.01130104, 0.01489519, 0.00320759, 0.01195869,
       0.02414779, 0.00359902, 0.01199982, 0.01114841, 0.00304532,
       0.01190761, 0.01188957, 0.00278619, 0.01229741, 0.05464491,
       0.01334717, 0.02787258, 0.01487243, 0.0144386 , 0.01288608,
       0.01370901, 0.02531659, 0.01213779, 0.14064917, 0.15110487,
       0.17001624, 0.02021508])

In [24]:
importance_indexes = np.where(importance > 0.01)
importance_indexes

(array([ 0,  1,  2,  3,  4,  5,  8, 10, 11, 12, 14, 16, 17, 19, 20, 22, 23,
        25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]),)

In [25]:
non_importance_indexes = np.where(importance < 0.01)
non_importance_indexes

(array([ 6,  7,  9, 13, 15, 18, 21, 24, 27]),)

In [26]:
selected_non_feature = transformer.get_feature_names_out()[non_importance_indexes]
selected_non_feature

array(['one hot encoding__PhoneService_No',
       'one hot encoding__PhoneService_Yes',
       'one hot encoding__MultipleLines_No phone service',
       'one hot encoding__InternetService_No',
       'one hot encoding__OnlineSecurity_No internet service',
       'one hot encoding__DeviceProtection_No internet service',
       'one hot encoding__TechSupport_No internet service',
       'one hot encoding__StreamingTV_No internet service',
       'one hot encoding__StreamingMovies_No internet service'],
      dtype=object)

We will remove the following colums from the feature dataset
<ul>
    <li>PhoneService</li>
    <li>InternetService</li>
</ul>

___

In [27]:
x_optimized = x.drop(columns=x.filter(regex="PhoneService|InternetService").columns)

In [28]:
x_optimized.head()

,one hot encoding__gender_Female,one hot encoding__gender_Male,one hot encoding__Partner_No,one hot encoding__Partner_Yes,one hot encoding__Dependents_No,one hot encoding__Dependents_Yes,one hot encoding__MultipleLines_No,one hot encoding__MultipleLines_No phone service,one hot encoding__MultipleLines_Yes,one hot encoding__OnlineSecurity_No,...,one hot encoding__PaperlessBilling_No,one hot encoding__PaperlessBilling_Yes,one hot encoding__PaymentMethod_Bank transfer (automatic),one hot encoding__PaymentMethod_Credit card (automatic),one hot encoding__PaymentMethod_Electronic check,one hot encoding__PaymentMethod_Mailed check,standard scaling__tenure,standard scaling__MonthlyCharges,standard scaling__TotalCharges,SeniorCitizen
0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-1.277445,-1.160323,-0.994971,0.0
1,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.066327,-0.259629,-0.173876,0.0
2,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,-1.236724,-0.362660,-0.960399,0.0
3,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.514251,-0.746535,-0.195400,0.0
4,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-1.236724,0.197365,-0.941193,0.0


Split the data for training and testing

In [35]:
x_train, x_test, y_train, y_test = train_test_split(x_optimized, y, random_state=42, shuffle=True, test_size=0.3)

In [36]:
estimator = XGBClassifier(objective='binary:logistic', random_state=42, eval_metric='logloss')

param_grid = {
    'max_depth':range(1, 30),
    'learning_rate': [0.1, 0.01, 0.05, 1, 10, 0.001],
    'n_estimators': range(10, 500, 25),
    'subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=50, random_state=42, shuffle=True)

model = GridSearchCV(estimator=estimator, cv=cv, param_grid=param_grid, n_jobs=20, refit=True, verbose=1,  scoring='roc_auc')

In [ ]:
model.fit(x_train, y_train)

Fitting 50 folds for each of 6960 candidates, totalling 348000 fits
